# 62 - Benchmark: Primer Dataset (Self Train-Test)

**Dataset:** Primer — `dataset_frontonly_conf60` (6,795 samples, 37 users, conf >= 60%).
**Split:** Sudah ada official user-wise 80/10/10 split (29/3/5 users).
**Model:** Semua model + Late Fusion (CNN, FCNN, Intermediate, CNN_TL, Intermediate_TL, Late_Fusion).
**Skenario:** B1 (Baseline) — konsisten dengan benchmark CK+/JAFFE/RAF-DB/KDEF.
**Metrik:** Macro F1, Micro F1 (=accuracy), Weighted F1.
**4-class:** remap 7→4 in-memory (neutral/happy/sad/negative).

In [1]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from training.models import (
    EmotionCNN, EmotionFCNN, IntermediateFusion,
    EmotionCNNTransfer, IntermediateFusionTransfer,
)
from training.utils import train_model, full_evaluation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 15
OUTPUT_DIR = PROJECT_ROOT / 'models' / 'benchmark' / 'primer'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODELS = [
    ('CNN', EmotionCNN, 'cnn', 0.0001),
    ('FCNN', EmotionFCNN, 'fcnn', 0.0001),
    ('Intermediate', IntermediateFusion, 'fusion', 0.0001),
    ('CNN_TL', EmotionCNNTransfer, 'cnn', 0.00005),
    ('Intermediate_TL', IntermediateFusionTransfer, 'fusion', 0.00005),
]

REMAP_4 = np.array([0, 1, 2, 3, 3, 3, 3], dtype=np.int64)  # 7 -> 4 class

print('Setup complete.')

Device: cuda
Setup complete.


In [2]:
# ── Helpers ──

def make_loader(images, landmarks, labels, model_type, batch_size=32, shuffle=True):
    img_t = torch.from_numpy(images).permute(0, 3, 1, 2)
    lm_t = torch.from_numpy(landmarks)
    y_t = torch.from_numpy(labels).long()
    if model_type == 'cnn': ds = TensorDataset(img_t, y_t)
    elif model_type == 'fcnn': ds = TensorDataset(lm_t, y_t)
    else: ds = TensorDataset(img_t, lm_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True)


def metrics_triple(y_true, y_pred):
    return {
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'micro_f1': float(f1_score(y_true, y_pred, average='micro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
    }


def train_and_eval(ModelClass, model_type, lr,
                   tr_img, tr_lm, tr_y, v_img, v_lm, v_y,
                   te_img, te_lm, te_y, emotions, save_dir):
    crit = nn.CrossEntropyLoss()
    save_path = str(save_dir / 'model.pth')
    test_loader = make_loader(te_img, te_lm, te_y, model_type, BATCH_SIZE, False)
    if (save_dir / 'model.pth').exists():
        print(f'    [SKIP] model.pth exists, loading and evaluating...')
        model = ModelClass(num_classes=len(emotions)).to(device)
        model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
        r = full_evaluation(model, test_loader, crit, device, model_type, emotions)
        return {'accuracy': float(r['test_accuracy']),
                'macro_f1': float(r['test_macro_f1']),
                'micro_f1': float(r['test_micro_f1']),
                'weighted_f1': float(r['test_weighted_f1'])}
    tr_loader = make_loader(tr_img, tr_lm, tr_y, model_type, BATCH_SIZE)
    val_loader = make_loader(v_img, v_lm, v_y, model_type, BATCH_SIZE, False)
    model = ModelClass(num_classes=len(emotions)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=8, min_lr=1e-7)
    train_model(model, tr_loader, val_loader, crit, opt, sch, device, model_type, EPOCHS, PATIENCE, save_path)
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    r = full_evaluation(model, test_loader, crit, device, model_type, emotions)
    return {'accuracy': float(r['test_accuracy']),
            'macro_f1': float(r['test_macro_f1']),
            'micro_f1': float(r['test_micro_f1']),
            'weighted_f1': float(r['test_weighted_f1'])}


def late_fusion_eval(tr_img, tr_lm, tr_y, v_img, v_lm, v_y,
                     te_img, te_lm, te_y, num_classes, save_dir):
    cnn = EmotionCNN(num_classes=num_classes).to(device)
    if (save_dir / 'cnn.pth').exists():
        print('    [SKIP] cnn.pth exists')
    else:
        tr_cnn = make_loader(tr_img, tr_lm, tr_y, 'cnn', BATCH_SIZE)
        val_cnn = make_loader(v_img, v_lm, v_y, 'cnn', BATCH_SIZE, False)
        opt = torch.optim.Adam(cnn.parameters(), lr=0.0001)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=8, min_lr=1e-7)
        train_model(cnn, tr_cnn, val_cnn, nn.CrossEntropyLoss(), opt, sch,
                    device, 'cnn', EPOCHS, PATIENCE, str(save_dir / 'cnn.pth'))

    fcnn = EmotionFCNN(num_classes=num_classes).to(device)
    if (save_dir / 'fcnn.pth').exists():
        print('    [SKIP] fcnn.pth exists')
    else:
        tr_fcnn = make_loader(tr_img, tr_lm, tr_y, 'fcnn', BATCH_SIZE)
        val_fcnn = make_loader(v_img, v_lm, v_y, 'fcnn', BATCH_SIZE, False)
        opt2 = torch.optim.Adam(fcnn.parameters(), lr=0.0001)
        sch2 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt2, mode='max', factor=0.5, patience=8, min_lr=1e-7)
        train_model(fcnn, tr_fcnn, val_fcnn, nn.CrossEntropyLoss(), opt2, sch2,
                    device, 'fcnn', EPOCHS, PATIENCE, str(save_dir / 'fcnn.pth'))

    cnn.load_state_dict(torch.load(save_dir / 'cnn.pth', map_location=device, weights_only=True))
    fcnn.load_state_dict(torch.load(save_dir / 'fcnn.pth', map_location=device, weights_only=True))
    cnn.eval(); fcnn.eval()

    def batched_softmax(model, arr, is_img, bs=32):
        outs = []
        with torch.no_grad():
            for i in range(0, len(arr), bs):
                chunk = arr[i:i+bs]
                if is_img:
                    t = torch.from_numpy(chunk).permute(0, 3, 1, 2).to(device)
                else:
                    t = torch.from_numpy(chunk).to(device)
                p = torch.softmax(model(t), dim=1).cpu().numpy()
                outs.append(p)
                del t
                torch.cuda.empty_cache()
        return np.concatenate(outs, axis=0)

    vc = batched_softmax(cnn, v_img, True)
    vf = batched_softmax(fcnn, v_lm, False)
    best_wf1, best_w = 0, 0.5
    for w in np.arange(0.0, 1.05, 0.05):
        pr = (w * vc + (1-w) * vf).argmax(axis=1)
        f = f1_score(v_y, pr, average='macro', zero_division=0)
        if f > best_wf1: best_wf1 = f; best_w = w
    print(f'    Late-fusion best w(cnn)={best_w:.2f}')
    cp = batched_softmax(cnn, te_img, True)
    fp = batched_softmax(fcnn, te_lm, False)
    preds = (best_w * cp + (1-best_w) * fp).argmax(axis=1)
    m = metrics_triple(te_y, preds)
    m['best_cnn_weight'] = float(best_w)
    return m


def run_benchmark(num_classes, emotions, y_tr, y_v, y_te,
                  tr_img, tr_lm, v_img, v_lm, te_img, te_lm):
    print(f"\n{'='*70}")
    print(f'  PRIMER {num_classes}-class (user-wise split, B1)')
    print(f"{'='*70}")
    print(f'  Train: {len(y_tr)}  Val: {len(y_v)}  Test: {len(y_te)}')
    all_results = {}
    for model_name, ModelClass, model_type, lr in MODELS + [('Late_Fusion', None, 'late', 0.0001)]:
        key = f'{model_name}_B1'
        print(f'\n  >> {key} ...')
        save_dir = OUTPUT_DIR / f'{num_classes}c' / key
        os.makedirs(save_dir, exist_ok=True)
        if model_type == 'late':
            r = late_fusion_eval(tr_img, tr_lm, y_tr, v_img, v_lm, y_v,
                                  te_img, te_lm, y_te, num_classes, save_dir)
        else:
            r = train_and_eval(ModelClass, model_type, lr,
                                tr_img, tr_lm, y_tr, v_img, v_lm, y_v,
                                te_img, te_lm, y_te, emotions, save_dir)
        all_results[key] = r
        print(f"    Macro={r['macro_f1']:.4f}  Micro={r['micro_f1']:.4f}  Weighted={r['weighted_f1']:.4f}")
    save_path = OUTPUT_DIR / f'primer_{num_classes}c_results.json'
    with open(save_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'\n  Saved: {save_path}')
    return all_results

print('Helper functions ready.')

Helper functions ready.


## Load Primer (conf60) dan Run

In [3]:
PRIMER_DIR = PROJECT_ROOT / 'data' / 'dataset_frontonly_conf60'
EMOTIONS_7 = ['neutral', 'happy', 'sad', 'angry', 'fearful', 'disgusted', 'surprised']
EMOTIONS_4 = ['neutral', 'happy', 'sad', 'negative']

tr_img = np.load(PRIMER_DIR / 'X_train_images.npy')
tr_lm = np.load(PRIMER_DIR / 'X_train_landmarks.npy')
y_tr_7 = np.load(PRIMER_DIR / 'y_train.npy')
v_img = np.load(PRIMER_DIR / 'X_val_images.npy')
v_lm = np.load(PRIMER_DIR / 'X_val_landmarks.npy')
y_v_7 = np.load(PRIMER_DIR / 'y_val.npy')
te_img = np.load(PRIMER_DIR / 'X_test_images.npy')
te_lm = np.load(PRIMER_DIR / 'X_test_landmarks.npy')
y_te_7 = np.load(PRIMER_DIR / 'y_test.npy')

# 4-class remap
y_tr_4 = REMAP_4[y_tr_7]
y_v_4 = REMAP_4[y_v_7]
y_te_4 = REMAP_4[y_te_7]

print(f'Primer (conf60) loaded: train={len(y_tr_7)}, val={len(y_v_7)}, test={len(y_te_7)}')
print(f'7-class train distribution: {Counter(y_tr_7.tolist())}')
print(f'4-class train distribution: {Counter(y_tr_4.tolist())}')

res_7c = run_benchmark(7, EMOTIONS_7, y_tr_7, y_v_7, y_te_7,
                       tr_img, tr_lm, v_img, v_lm, te_img, te_lm)
res_4c = run_benchmark(4, EMOTIONS_4, y_tr_4, y_v_4, y_te_4,
                       tr_img, tr_lm, v_img, v_lm, te_img, te_lm)

Primer (conf60) loaded: train=5287, val=579, test=929
7-class train distribution: Counter({0: 4526, 1: 416, 2: 287, 3: 27, 6: 16, 5: 13, 4: 2})
4-class train distribution: Counter({0: 4526, 1: 416, 2: 287, 3: 58})

  PRIMER 7-class (user-wise split, B1)
  Train: 5287  Val: 579  Test: 929

  >> CNN_B1 ...


 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      1.1195     0.6966     0.9500    0.8256   0.1406   0.000100  (33.4s)


     2      0.5711     0.8587     0.8413    0.8204   0.1484   0.000100  (33.1s)


     3      0.4630     0.8659     0.7248    0.7876   0.1468   0.000100  (33.8s)


...

    43      0.3500     0.8788     0.7576    0.8480   0.3598   0.000025  (0.6s)


    44      0.3458     0.8757     0.6604    0.8377   0.3535   0.000025  (0.7s)


    45      0.3402     0.8873     0.5737    0.8290   0.4448   0.000025  (0.6s)

Early stopping at epoch 45. Best epoch: 30 (val_f1=0.4761)

Best: epoch 30, val_acc=0.8204, val_f1=0.4761
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/primer/4c/Late_Fusion_B1/fcnn.pth


    Late-fusion best w(cnn)=0.20


    Macro=0.4600  Micro=0.7632  Weighted=0.7793

  Saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/primer/primer_4c_results.json


## Ringkasan Primer (Semua Metrik)

In [4]:
for nc_label, res in [('7-class', res_7c), ('4-class', res_4c)]:
    print(f"\n{'='*78}")
    print(f'  PRIMER {nc_label} - sorted by Macro F1')
    print(f"{'='*78}")
    print(f"  {'Model':<22} {'Macro':>10} {'Micro':>10} {'Weighted':>10} {'Accuracy':>10}")
    print(f"  {'-'*70}")
    for key in sorted(res.keys(), key=lambda k: -res[k]['macro_f1']):
        r = res[key]
        print(f"  {key:<22} {r['macro_f1']:>10.4f} {r['micro_f1']:>10.4f} {r['weighted_f1']:>10.4f} {r['accuracy']:>10.4f}")


  PRIMER 7-class - sorted by Macro F1
  Model                       Macro      Micro   Weighted   Accuracy
  ----------------------------------------------------------------------
  Intermediate_TL_B1         0.2922     0.8084     0.8189     0.8084
  CNN_TL_B1                  0.2805     0.8192     0.8144     0.8192
  CNN_B1                     0.2698     0.7869     0.7953     0.7869
  FCNN_B1                    0.2608     0.7664     0.7921     0.7664
  Intermediate_B1            0.2596     0.7987     0.7931     0.7987
  Late_Fusion_B1             0.2443     0.7783     0.7767     0.7783

  PRIMER 4-class - sorted by Macro F1
  Model                       Macro      Micro   Weighted   Accuracy
  ----------------------------------------------------------------------
  Intermediate_TL_B1         0.4822     0.7804     0.7902     0.7804
  CNN_B1                     0.4755     0.8030     0.8017     0.8030
  Late_Fusion_B1             0.4600     0.7632     0.7793     0.7632
  FCNN_B1        